In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


### 1. Load Extracted Pages

In [2]:
# Imports and file paths
from pathlib import Path
from collections import Counter
import re
import unicodedata
import numpy as np
import pandas as pd

PROJECT_DIR = Path('/content/drive/MyDrive/NLP Project')  # Change if needed
DATA_DIR = PROJECT_DIR / 'data'
INPUT_FILE = DATA_DIR / 'extracted_text' / 'extracted_pages.jsonl'
OUTPUT_DIR = DATA_DIR / 'preprocessed_text'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / 'preprocessed_pages.jsonl'
EXCLUDED_FILE = OUTPUT_DIR / 'excluded_pages.csv'
ARTICLE_REVIEW_FILE = OUTPUT_DIR / 'article_review.csv'
HEADER_LOG_FILE = OUTPUT_DIR / 'removed_header_footer_lines.csv'
MIN_WORDS = 25  # Short pages stay in the output but are not indexed

In [4]:
# Load extracted pages and check required fields
raw_df = pd.read_json(INPUT_FILE, lines=True)
required = {'article_id', 'page_number', 'text'}
missing = required - set(raw_df.columns)
if missing:
    raise ValueError(f'Missing required columns: {sorted(missing)}')

raw_df['text'] = raw_df['text'].fillna('').astype(str)
raw_df = raw_df.sort_values(['article_id', 'page_number']).reset_index(drop=True)
print(f"Loaded {len(raw_df)} pages from {raw_df['article_id'].nunique()} papers")

Loaded 3619 pages from 182 papers


### 2. Clean text conservatively

In [5]:
# Markers that often indicate failed character extraction
ERROR_MARKER = re.compile(r'\ufffd|\(cid:\d+\)')
CONTROL_CHARS = re.compile(r'[\x00-\x08\x0b-\x1f\x7f-\x9f\ue000-\uf8ff]')
INVISIBLE_CHARS = dict.fromkeys(map(ord, '\u200b\u200c\u200d\u2060\ufeff\u00ad'))

# Recognizable publisher boilerplate and table-of-contents dotted leaders
BOILERPLATE = re.compile(
    r'^(downloaded from|contents lists available|journal homepage|check for updates|'
    r'production and hosting by|peer review under responsibility)', re.I
)
DOT_LEADER = re.compile(r'(?:\.{2,}|\.\s){3,}')


def clean_text(text):
    """Normalize characters and remove obvious extraction noise, not meaningful content."""
    text = unicodedata.normalize('NFKC', text).translate(INVISIBLE_CHARS)
    text = ERROR_MARKER.sub(' ', text)
    text = CONTROL_CHARS.sub(' ', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r' ?\n ?', '\n', text)
    text = re.sub(r'\n{3,}', '\n\n', text)

    kept_lines = []
    for line in text.splitlines():
        if BOILERPLATE.match(line.strip()):
            continue
        if DOT_LEADER.search(line):
            continue
        kept_lines.append(line)
    return '\n'.join(kept_lines).strip()

# Count extraction markers before cleaning removes them
raw_df['bad_chars'] = raw_df['text'].str.count(ERROR_MARKER)
raw_df['clean_text'] = raw_df['text'].map(clean_text)

In [6]:
# Repair words split by a PDF line break, for example "inspe-\nct" -> "inspect".
# All changes are logged because some real compound words may also be hyphenated at line ends.
LINEBREAK_HYPHEN = re.compile(r'([A-Za-z]{2,})-\n([a-z]{2,})')

def repair_linebreak_hyphens(text):
    repairs = []
    def join_word(match):
        fixed = match.group(1) + match.group(2)
        repairs.append((match.group(0), fixed))
        return fixed
    repaired_text = LINEBREAK_HYPHEN.sub(join_word, text)
    return repaired_text, repairs

raw_df['hyphen_repairs'] = ''
for row_index, text in raw_df['clean_text'].items():
    fixed_text, repairs = repair_linebreak_hyphens(text)
    raw_df.at[row_index, 'clean_text'] = fixed_text
    if repairs:
        raw_df.at[row_index, 'hyphen_repairs'] = '; '.join(
            f'{old!r} -> {new}' for old, new in repairs
        )

### 3. Remove repeated headers and footers

In [7]:
# Only consider lines within the first/last three lines of each page.
# A repeated line in the middle of a page is never removed by this function.
PAGE_NUMBER = re.compile(r'^(?:page\s*)?\d+(?:\s*(?:of|/)\s*\d+)?$', re.I)

def normalize_line(line):
    """Normalize digits so page numbers differ only by their number."""
    line = re.sub(r'\d+', '#', line.lower())
    return re.sub(r'\s+', ' ', line).strip()


def remove_headers_footers(page_texts, article_id):
    if len(page_texts) < 3:
        return page_texts, []

    # Count each candidate at most once per page; only look at edge lines.
    edge_counts = Counter()
    for text in page_texts:
        lines = text.splitlines()
        edge_lines = lines[:3] + lines[-3:]
        edge_counts.update({normalize_line(x) for x in edge_lines if x.strip()})

    min_repeats = max(3, int(np.ceil(0.30 * len(page_texts))))
    repeated = {line for line, count in edge_counts.items() if count >= min_repeats}
    cleaned_pages, log = [], []

    for page_number, text in enumerate(page_texts):
        lines = text.splitlines()
        kept = []
        for i, line in enumerate(lines):
            at_edge = i < 3 or i >= len(lines) - 3
            normalized = normalize_line(line)
            is_page_number = bool(PAGE_NUMBER.match(line.strip()))
            is_repeated_header = normalized in repeated and len(line.strip()) >= 12

            if at_edge and (is_page_number or is_repeated_header):
                log.append({'article_id': article_id, 'line': line.strip()})
            else:
                kept.append(line)
        cleaned_pages.append('\n'.join(kept).strip())
    return cleaned_pages, log

removed_lines = []
for article_id, group in raw_df.groupby('article_id', sort=False):
    cleaned, log = remove_headers_footers(group['clean_text'].tolist(), article_id)
    raw_df.loc[group.index, 'clean_text'] = cleaned
    removed_lines.extend(log)

header_log = pd.DataFrame(removed_lines, columns=['article_id', 'line'])
print('Header/footer lines removed:', len(header_log))

Header/footer lines removed: 8587


### 4. Flag incomplete or low-quality documents

In [8]:
# Detect page-number gaps when page_number is numeric.
# A non-1 starting page is flagged as a possible partial document, not proof of one.
def inspect_article(group):
    values = pd.to_numeric(group['page_number'], errors='coerce').dropna().astype(int).tolist()
    values = sorted(set(values))
    gaps = []
    if len(values) >= 2:
        gaps = [n for n in range(values[0], values[-1] + 1) if n not in values]

    flags = []
    if gaps:
        flags.append('missing_page_numbers')
    if values and values[0] > 1:
        flags.append('starts_after_page_1_possible_partial')
    if (group['clean_text'].str.strip() == '').any():
        flags.append('one_or_more_empty_pages')
    if (group['bad_chars'] > 0).any():
        flags.append('extraction_error_markers')

    return pd.Series({
        'page_count': len(group),
        'first_numeric_page': values[0] if values else np.nan,
        'last_numeric_page': values[-1] if values else np.nan,
        'missing_page_numbers': ','.join(map(str, gaps)),
        'article_review_flags': ';'.join(flags)
    })

article_audit = raw_df.groupby('article_id', sort=False).apply(inspect_article).reset_index()
print('Papers flagged for review:', int(article_audit['article_review_flags'].ne('').sum()))
article_audit.head()

Papers flagged for review: 0


/tmp/ipykernel_2175/3795812154.py:28: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  article_audit = raw_df.groupby('article_id', sort=False).apply(inspect_article).reset_index()


,article_id,page_count,first_numeric_page,last_numeric_page,missing_page_numbers,article_review_flags
0,FD_001,30,1,30,,
1,FD_002,11,1,11,,
2,FD_003,8,1,8,,
3,FD_004,29,1,29,,
4,FD_005,17,1,17,,


### 5. Handle reference lists conservatively

In [9]:
# Remove a bibliography only when a clear heading in the latter part of a paper
# is followed by citation evidence. Record the method for later manual checking.
REFERENCE_HEADING = re.compile(
    r'^\s*(?:(?:\d{1,2}|[IVX]{1,5})\.?\s*)?'
    r'(?:references?|bibliography|works cited|literature cited|reference list)\s*:?[ \t]*$',
    re.I | re.M
)
YEAR = re.compile(r'\b(?:19\d{2}|20\d{2})\b')
CITATION_HINT = re.compile(r'doi|https?://|\bet al\b|journal|proceedings|conference|arxiv|springer|elsevier', re.I)


def find_reference_start(page_texts):
    """Return page index and character position for a supported references heading."""
    total = len(page_texts)
    for page_i in range(total - 1, -1, -1):
        if (page_i + 1) / max(total, 1) < 0.40:
            break
        for match in reversed(list(REFERENCE_HEADING.finditer(page_texts[page_i]))):
            following = page_texts[page_i][match.end():] + '\n' + '\n'.join(page_texts[page_i + 1:page_i + 3])
            has_citation_evidence = len(YEAR.findall(following)) >= 2 and bool(CITATION_HINT.search(following))
            if has_citation_evidence:
                return page_i, match.start()
    return None

raw_df['status'] = 'body'
raw_df['reference_method'] = 'none'
raw_df['reference_review'] = False
for article_id, group in raw_df.groupby('article_id', sort=False):
    indices = group.index.tolist()
    page_texts = group['clean_text'].tolist()
    start = find_reference_start(page_texts)

    if start is None:
        # No automatic deletion when no supported heading is found.
        raw_df.loc[indices, 'reference_review'] = True
        continue

    page_i, char_i = start
    page_texts[page_i] = page_texts[page_i][:char_i]
    raw_df.loc[indices, 'reference_method'] = 'heading_with_citation_evidence'
    for later_i in range(page_i + 1, len(indices)):
        page_texts[later_i] = ''
        raw_df.loc[indices[later_i], 'status'] = 'reference_list'
    if not page_texts[page_i].strip():
        raw_df.loc[indices[page_i], 'status'] = 'reference_list'
    raw_df.loc[indices, 'clean_text'] = page_texts

# Word counts and explicit indexing decisions
raw_df['n_words'] = raw_df['clean_text'].str.split().str.len()
raw_df['drop_reason'] = ''
raw_df.loc[raw_df['status'] == 'reference_list', 'drop_reason'] = 'reference_list'
short_pages = (raw_df['n_words'] < MIN_WORDS) & raw_df['drop_reason'].eq('')
raw_df.loc[short_pages, 'drop_reason'] = 'too_short'
raw_df['keep_for_index'] = raw_df['drop_reason'].eq('')

6. Detect and exclude duplicate research papers

In [10]:
# STEP 6: DETECT AND EXCLUDE DUPLICATE RESEARCH PAPERS

import re
from itertools import combinations

# Record which original article each duplicate belongs to
raw_df["duplicate_of"] = pd.NA


# 1. NORMALIZE ARTICLE TITLES

def normalize_title(title):
    """Normalize a title for duplicate comparison."""

    if pd.isna(title):
        return ""

    title = str(title).lower()

    title = re.sub(r"[^a-z0-9]+", " ", title)

    return title.strip()


# 2. BUILD TEXT SHINGLES FOR DOCUMENT COMPARISON

def get_article_shingles(article_id, n=5):
    """
    Create word shingles from all cleaned pages
    belonging to a research article.
    """

    pages = raw_df.loc[
        raw_df["article_id"] == article_id
    ].sort_values("page_number")

    article_text = " ".join(
        pages["clean_text"].fillna("").astype(str)
    )

    tokens = re.findall(
        r"[a-z0-9]+",
        article_text.lower()
    )

    if len(tokens) < n:
        return set()

    return {
        tuple(tokens[i:i+n])
        for i in range(len(tokens) - n + 1)
    }


# 3. JACCARD SIMILARITY

def jaccard_similarity(set_a, set_b):

    if not set_a or not set_b:
        return 0.0

    intersection = len(set_a & set_b)
    union = len(set_a | set_b)

    return intersection / union


# 4. PREPARE ARTICLE METADATA

article_info = (
    raw_df.groupby("article_id", as_index=False)
    .agg(
        title=("title", "first"),
        year=("year", "first"),
        page_count=("page_number", "count")
    )
)

article_info["normalized_title"] = (
    article_info["title"].map(normalize_title)
)

article_info["year_key"] = (
    article_info["year"].astype("string").str.strip()
)


# 5. DETECT DUPLICATES

duplicate_map = {}
duplicate_records = []

SIMILARITY_THRESHOLD = 0.98

for (_, _), group in article_info.groupby(
    ["normalized_title", "year_key"],
    dropna=False
):

    # Ignore missing titles
    if not group["normalized_title"].iloc[0]:
        continue

    article_ids = sorted(group["article_id"].tolist())

    if len(article_ids) < 2:
        continue

    # Compare papers sharing the same normalized title and year
    for article_a, article_b in combinations(article_ids, 2):

        shingles_a = get_article_shingles(article_a)
        shingles_b = get_article_shingles(article_b)

        similarity = jaccard_similarity(
            shingles_a,
            shingles_b
        )

        if similarity >= SIMILARITY_THRESHOLD:

            # Keep the first ID as the canonical article
            # unless it has already been marked as a duplicate.
            canonical = duplicate_map.get(
                article_a,
                article_a
            )

            duplicate_map[article_b] = canonical

            duplicate_records.append({
                "kept_article_id": canonical,
                "duplicate_article_id": article_b,
                "similarity": round(similarity, 6)
            })


# 6. EXCLUDE DUPLICATES FROM INDEXING

for duplicate_id, kept_id in duplicate_map.items():

    mask = raw_df["article_id"] == duplicate_id

    raw_df.loc[mask, "duplicate_of"] = kept_id

    raw_df.loc[
        mask,
        "drop_reason"
    ] = "duplicate_article"

    raw_df.loc[
        mask,
        "keep_for_index"
    ] = False


# 7. CREATE DUPLICATE REPORT

duplicate_report = pd.DataFrame(
    duplicate_records,
    columns=[
        "kept_article_id",
        "duplicate_article_id",
        "similarity"
    ]
)

# Include duplicate information in the article audit
article_audit["duplicate_of"] = (
    article_audit["article_id"].map(duplicate_map)
)

DUPLICATE_LOG_FILE = (
    OUTPUT_DIR / "duplicate_articles.csv"
)

duplicate_report.to_csv(
    DUPLICATE_LOG_FILE,
    index=False
)


# 8. VALIDATE RESULTS

assert not raw_df.loc[
    raw_df["duplicate_of"].notna(),
    "keep_for_index"
].any()

print("=" * 60)
print("ARTICLE DEDUPLICATION SUMMARY")
print("=" * 60)

print("Original articles:", article_info["article_id"].nunique())
print("Duplicate articles:", len(duplicate_map))

print(
    "Unique articles retained:",
    article_info["article_id"].nunique() - len(duplicate_map)
)

print("\nDetected duplicate papers:")
display(duplicate_report)

print("\nDuplicate report saved to:")
print(DUPLICATE_LOG_FILE)

ARTICLE DEDUPLICATION SUMMARY
Original articles: 182
Duplicate articles: 2
Unique articles retained: 180

Detected duplicate papers:


,kept_article_id,duplicate_article_id,similarity
0,FD_104,FD_172,1.0
1,FD_072,FD_117,1.0



Duplicate report saved to:
/content/drive/MyDrive/NLP Project/data/preprocessed_text/duplicate_articles.csv


7. Validate, inspect, and save

In [11]:

# STEP 7: VALIDATE, INSPECT AND SAVE PREPROCESSED DATA

# 1. Confirm page records and citation metadata are retained

assert len(raw_df) == len(
    pd.read_json(INPUT_FILE, lines=True)
)

assert raw_df["article_id"].notna().all()
assert raw_df["page_number"].notna().all()
assert raw_df["clean_text"].notna().all()

assert raw_df["keep_for_index"].eq(
    raw_df["drop_reason"].eq("")
).all()


# 2. VERIFY DUPLICATE ARTICLES ARE EXCLUDED FROM INDEXING

assert "duplicate_of" in raw_df.columns, (
    "Run the article deduplication cell before this cell."
)

duplicate_pages = raw_df["duplicate_of"].notna()

assert not raw_df.loc[
    duplicate_pages, "keep_for_index"
].any(), "Some duplicate articles are still marked for indexing."

print("Duplicate article validation passed.")

print(
    "Duplicate articles excluded:",
    raw_df.loc[
        duplicate_pages, "article_id"
    ].nunique()
)

print(
    "Articles eligible for indexing:",
    raw_df.loc[
        raw_df["keep_for_index"], "article_id"
    ].nunique()
)


# 3. SHOW RAW AND CLEANED EXAMPLES

for _, row in raw_df.sample(
    n=min(5, len(raw_df)),
    random_state=42
).iterrows():

    print(
        "ARTICLE:", row["article_id"],
        "| PAGE:", row["page_number"],
        "| STATUS:", row["status"],
        "| KEEP:", row["keep_for_index"],
        "| DUPLICATE OF:", row["duplicate_of"]
    )

    print("RAW TEXT:", str(row["text"])[:500])
    print("CLEAN TEXT:", str(row["clean_text"])[:500])
    print("-" * 70)


# 4. DEFINE METADATA COLUMNS

metadata_cols = [
    c for c in [
        "article_id",
        "title",
        "authors",
        "year",
        "venue",
        "doi",
        "page_number",
        "page_label"
    ]
    if c in raw_df.columns
]


# 5. DEFINE OUTPUT COLUMNS

output_cols = metadata_cols + [
    c for c in [
        "clean_text",
        "n_words",
        "bad_chars",
        "status",
        "reference_method",
        "reference_review",
        "drop_reason",
        "keep_for_index",
        "duplicate_of"
    ]
    if c in raw_df.columns
]


# 6. SAVE ALL PREPROCESSED PAGE RECORDS

# Preserve all pages for auditing.
# Duplicate articles have keep_for_index=False.
# The chunking notebook must filter on keep_for_index.

raw_df[output_cols].to_json(
    OUTPUT_FILE,
    orient="records",
    lines=True,
    force_ascii=False
)


# 7. SAVE EXCLUDED PAGE RECORDS

raw_df.loc[
    ~raw_df["keep_for_index"],
    metadata_cols + [
        "drop_reason",
        "n_words",
        "duplicate_of"
    ]
].to_csv(
    EXCLUDED_FILE,
    index=False
)


# 8. SAVE ARTICLE AUDIT

article_audit.to_csv(
    ARTICLE_REVIEW_FILE,
    index=False
)


# 9. SAVE REMOVED HEADER LOG

if not header_log.empty:

    header_log.to_csv(
        HEADER_LOG_FILE,
        index=False
    )


# 10. FINAL SUMMARY

print("\n" + "=" * 60)
print("PREPROCESSING COMPLETED")
print("=" * 60)

print("Total page records:", len(raw_df))

print(
    "Original articles:",
    raw_df["article_id"].nunique()
)

print(
    "Duplicate articles:",
    raw_df.loc[
        duplicate_pages, "article_id"
    ].nunique()
)

print(
    "Articles eligible for indexing:",
    raw_df.loc[
        raw_df["keep_for_index"], "article_id"
    ].nunique()
)

print(
    "Pages eligible for indexing:",
    int(raw_df["keep_for_index"].sum())
)

print("\nSaved preprocessed pages:", OUTPUT_FILE)
print("Saved excluded-page log:", EXCLUDED_FILE)
print("Saved article completeness review:", ARTICLE_REVIEW_FILE)
print("Saved removed-header log:", HEADER_LOG_FILE)


Duplicate article validation passed.
Duplicate articles excluded: 2
Articles eligible for indexing: 180
ARTICLE: FD_125 | PAGE: 79 | STATUS: reference_list | KEEP: False | DUPLICATE OF: <NA>
RAW TEXT: 4243
Machine Learning (2024) 113:4165–4243	
1 3
	
Bartosz Krawczyk 
	
bkrawczyk@vcu.edu
1	
Department of Computer Science, Virginia Commonwealth University, 401 W. Main St. ERB2334, 
Richmond, VA 23284, USA
2	
Department of Computer Science, Virginia Commonwealth University, 401 W. Main St. ERB2316, 
Richmond, VA 23284, USA
3	
Department of Computer Science, Virginia Commonwealth University, 401 W. Main St. ERB2314, 
Richmond, VA 23284, USA
CLEAN TEXT: 
----------------------------------------------------------------------
ARTICLE: FD_117 | PAGE: 3 | STATUS: body | KEEP: False | DUPLICATE OF: FD_072
RAW TEXT: CUI ET AL.: FRAUDGNN-RL: A GRAPH NEURAL NETWORK WITH REINFORCEMENT LEARNING FOR ADAPTIVE FINANCIAL FRAUD DETECTION
through adaptive learning, though their solution still requires
per